# Lesson 32: Semi-supervised & Active Learning

*ML Academy — companion notebook*

Learning with few labels: self-training and pseudo-labels, label spreading on a graph, and active learning with uncertainty sampling.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-32.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. The baseline: 50 labels

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
rng = np.random.default_rng(0)
labelled = rng.choice(len(X_pool), 50, replace=False)            # only 50 of 1,257 images labelled
clf = LogisticRegression(max_iter=2000).fit(X_pool[labelled], y_pool[labelled])
full = LogisticRegression(max_iter=2000).fit(X_pool, y_pool)
print("labelled examples:", len(labelled), "of", len(X_pool))
print("accuracy with 50 labels:", round(clf.score(X_test, y_test), 3), "  with all labels:", round(full.score(X_test, y_test), 3))

labelled examples: 50 of 1257
accuracy with 50 labels: 0.811   with all labels: 0.972


## 4. Self-training with pseudo-labels

In [2]:
from sklearn.semi_supervised import SelfTrainingClassifier
y_partial = np.full(len(y_pool), -1); y_partial[labelled] = y_pool[labelled]   # -1 means "unlabelled"
for thr in [0.99, 0.9, 0.6]:
    st = SelfTrainingClassifier(LogisticRegression(max_iter=2000), threshold=thr).fit(X_pool, y_partial)
    added = (st.labeled_iter_ > 0).sum()
    wrong = (st.transduction_[st.labeled_iter_ > 0] != y_pool[st.labeled_iter_ > 0]).mean()
    print(f"threshold {thr}: pseudo-labels added {added}, of which wrong {wrong:.1%}, test accuracy {st.score(X_test, y_test):.3f}")

threshold 0.99: pseudo-labels added 66, of which wrong 1.5%, test accuracy 0.819


threshold 0.9: pseudo-labels added 1008, of which wrong 5.8%, test accuracy 0.911


threshold 0.6: pseudo-labels added 1189, of which wrong 13.0%, test accuracy 0.872


## 5. Label spreading on a graph

In [3]:
from sklearn.semi_supervised import LabelSpreading
ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2).fit(X_pool, y_partial)
spread_acc = (ls.transduction_ == y_pool).mean()
print("label spreading: share of pool labels inferred correctly", round(spread_acc, 3))
clf_ls = LogisticRegression(max_iter=2000).fit(X_pool, ls.transduction_)
print("logistic regression trained on the spread labels: test accuracy", round(clf_ls.score(X_test, y_test), 3))

label spreading: share of pool labels inferred correctly 0.876
logistic regression trained on the spread labels: test accuracy 0.881


## 6. Active learning: choose what to label

In [4]:
def active_learning(strategy, rounds=10, batch=15, seed=0):
    r = np.random.default_rng(seed)
    lab = list(r.choice(len(X_pool), 20, replace=False))
    for _ in range(rounds):
        m = LogisticRegression(max_iter=2000).fit(X_pool[lab], y_pool[lab])
        rest = np.setdiff1d(np.arange(len(X_pool)), lab)
        if strategy == "uncertainty":
            P = np.sort(m.predict_proba(X_pool[rest]), axis=1)
            pick = rest[np.argsort(P[:, -1] - P[:, -2])[:batch]]    # smallest margin between the top two classes
        else:
            pick = r.choice(rest, batch, replace=False)
        lab += list(pick)
    return len(lab), LogisticRegression(max_iter=2000).fit(X_pool[lab], y_pool[lab]).score(X_test, y_test)
for strategy in ["random", "uncertainty"]:
    res = [active_learning(strategy, seed=s) for s in range(5)]
    print(f"{strategy:11s}: {res[0][0]} labels, mean test accuracy {np.mean([a for _, a in res]):.3f}")

random     : 170 labels, mean test accuracy 0.917


uncertainty: 170 labels, mean test accuracy 0.947


In [5]:
budgets = [50, 100, 200, 400]
for b in budgets:
    accs = []
    for s in range(3):
        idx = np.random.default_rng(s).choice(len(X_pool), b, replace=False)
        accs.append(LogisticRegression(max_iter=2000).fit(X_pool[idx], y_pool[idx]).score(X_test, y_test))
    print(f"{b} random labels: test accuracy {np.mean(accs):.3f}")

50 random labels: test accuracy 0.814
100 random labels: test accuracy 0.886
200 random labels: test accuracy 0.922
400 random labels: test accuracy 0.946


## 9. In Python: self-training by hand

In [6]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_pool, X_test, y_pool, y_test = train_test_split(X / 16, y, test_size=0.3, random_state=1, stratify=y)
lab = np.random.default_rng(1).choice(len(X_pool), 60, replace=False)
Xl, yl = X_pool[lab], y_pool[lab]
Xu = np.delete(X_pool, lab, axis=0)
for rnd in range(1, 5):
    m = LogisticRegression(max_iter=3000).fit(Xl, yl)
    P = m.predict_proba(Xu)
    sure = P.max(axis=1) > 0.9
    print(f"round {rnd}: {len(yl)} training labels, test accuracy {m.score(X_test, y_test):.3f}, adding {sure.sum()} pseudo-labels")
    Xl, yl = np.vstack([Xl, Xu[sure]]), np.concatenate([yl, m.classes_[P[sure].argmax(axis=1)]])
    Xu = Xu[~sure]

round 1: 60 training labels, test accuracy 0.878, adding 14 pseudo-labels


round 2: 74 training labels, test accuracy 0.876, adding 32 pseudo-labels
round 3: 106 training labels, test accuracy 0.870, adding 47 pseudo-labels
round 4: 153 training labels, test accuracy 0.857, adding 36 pseudo-labels


## 12. Going deeper: when unlabelled data misleads, and committees

In [7]:
import numpy as np
from sklearn.semi_supervised import LabelSpreading
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
# The label depends on feature 1; feature 2 is irrelevant noise on a scale 50 times larger.
X = np.c_[rng.normal(0, 1, 600), rng.normal(0, 50, 600)]
y = (X[:, 0] > 0).astype(int)
lab = rng.choice(600, 20, replace=False)
yp = np.full(600, -1); yp[lab] = y[lab]
test = np.setdiff1d(np.arange(600), lab)
sup = LogisticRegression().fit(X[lab], y[lab]).score(X[test], y[test])
spread = (LabelSpreading(kernel="knn", n_neighbors=10).fit(X, yp).transduction_[test] == y[test]).mean()
Xs = X / X.std(axis=0)
spread_s = (LabelSpreading(kernel="knn", n_neighbors=10).fit(Xs, yp).transduction_[test] == y[test]).mean()
print("labelled only:", round(sup, 3), "  label spreading, raw features:", round(spread, 3), "  after scaling:", round(spread_s, 3))

labelled only: 0.919   label spreading, raw features: 0.557   after scaling: 0.864


In [8]:
from sklearn.datasets import load_digits
from sklearn.ensemble import BaggingClassifier
Xd, yd = load_digits(return_X_y=True); Xd = Xd / 16
pool, test2 = np.arange(1200), np.arange(1200, len(yd))
def run(strategy, seed):
    r = np.random.default_rng(seed); lab = list(r.choice(pool, 20, replace=False))
    for _ in range(10):
        rest = np.setdiff1d(pool, lab)
        if strategy == "committee":
            com = BaggingClassifier(LogisticRegression(max_iter=1000), n_estimators=5, random_state=seed).fit(Xd[lab], yd[lab])
            votes = np.array([est.predict(Xd[rest]) for est in com.estimators_])
            disagree = np.array([len(set(v)) for v in votes.T])           # number of distinct votes
            pick = rest[np.argsort(-disagree + 1e-3 * r.random(len(rest)))[:15]]
        else:
            pick = r.choice(rest, 15, replace=False)
        lab += list(pick)
    return LogisticRegression(max_iter=1000).fit(Xd[lab], yd[lab]).score(Xd[test2], yd[test2])
for s in ["random", "committee"]:
    print(f"{s:9s}: test accuracy with 170 labels {np.mean([run(s, k) for k in range(3)]):.3f}")

random   : test accuracy with 170 labels 0.883


committee: test accuracy with 170 labels 0.886


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(df[["size_m2", "dist_km", "age_yrs", "price_lakh"]])
known = np.array([True, False, False, True, False, True, False, False, True, False, True, False])
y = np.where(known, df.sold_fast, -1)
ls = LabelSpreading(kernel="knn", n_neighbors=3).fit(X, y)
hidden = ~known
print("labels known for", known.sum(), "homes; inferred for", hidden.sum())
print("inferred correctly:", int((ls.transduction_[hidden] == df.sold_fast[hidden]).sum()), "of", hidden.sum())

labels known for 5 homes; inferred for 7
inferred correctly: 4 of 7


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Margin uncertainty

Compute the margin between the two highest probabilities.

In [11]:
import numpy as np
p = np.array([0.2, 0.45, 0.35])
margin = p.max()  # ← subtract the second-highest probability
check(margin, 0.1, hint='Sort descending, then subtract the second from the first.')

❌ Not yet. Hint: Sort descending, then subtract the second from the first.


### Challenge 2: Pick the most uncertain

Return the index of the row with the smallest margin.

In [12]:
import numpy as np
P = np.array([[0.9, 0.1], [0.55, 0.45], [0.7, 0.3]])
pick = int(np.argmax(P[:, 0]))  # ← that is the most confident
check(pick, 1, hint='The margin is the gap between the two probabilities; take the argmin.')

❌ Not yet. Hint: The margin is the gap between the two probabilities; take the argmin.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Margin uncertainty** — the missing line:

```python
margin = round(float(np.sort(p)[-1] - np.sort(p)[-2]), 2)
```

**2. Pick the most uncertain** — the missing line:

```python
pick = int(np.argmin(np.abs(P[:, 0] - P[:, 1])))
```